In [2]:
import numpy as np
import polars as pl
import time
import warnings
warnings.filterwarnings('ignore')

from rustima import SARIMAXModel, auto_arima

from statsmodels.tsa.statespace.sarimax import SARIMAX as SM_SARIMAX
import pmdarima as pm

메모리 타입체크(e.g. 사용량 속도) 측정해봐요

In [3]:
# 열수요 데이터
df = pl.read_csv("/Users/icy71/GitHub/Rust-python-arima/rustima/0_Example_cy/total_heat_demand_dwh.csv")
print(f"\n시간단위 데이터: {df.shape}")
print(df.head())


시간단위 데이터: (473022, 22)
shape: (5, 22)
┌───────────┬───────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ date      ┆ time_inde ┆ branch_id ┆ temperatu ┆ … ┆ day_name ┆ day_name_ ┆ is_weeken ┆ is_holida │
│ ---       ┆ x         ┆ ---       ┆ re        ┆   ┆ ---      ┆ kr        ┆ d         ┆ y         │
│ str       ┆ ---       ┆ str       ┆ ---       ┆   ┆ str      ┆ ---       ┆ ---       ┆ ---       │
│           ┆ i64       ┆           ┆ f64       ┆   ┆          ┆ str       ┆ i64       ┆ i64       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 2021-01-0 ┆ 202101010 ┆ 중앙      ┆ -10.1     ┆ … ┆ Friday   ┆ 금        ┆ 0         ┆ 1         │
│ 1         ┆ 1         ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ 01:00:00  ┆           ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ 2021-01-0 ┆ 202101010 ┆ 파주      ┆ -11.5     ┆ … ┆ Fri

In [4]:
# 파주 지점만 필터링 + 날짜 파싱 + 정렬
df_paju = (
    df.filter(pl.col('branch_id') == '파주')
      .with_columns(pl.col('date').str.to_datetime())
      .sort('date')
)

# 시간당 빈도로 재인덱싱해서 누락된 시점 확인 (hourly)
full_idx = pl.datetime_range(
    df_paju['date'].min(),
    df_paju['date'].max(),
    interval='1h',
    eager=True,
)
missing_hours = full_idx.filter(~full_idx.is_in(df_paju['date']))

print(f"행 수: {df_paju.height:,}  |  누락 시점: {missing_hours.len():,}개")

행 수: 26,279  |  누락 시점: 0개


In [5]:
y_paju = df_paju['heat_demand']

## auto_arima 속도 비교 — rustima vs pmdarima

같은 데이터(`y_paju`의 subset) · 같은 탐색 방식(stepwise) · 같은 기준(AIC)으로 두 엔진 탐색 시간 비교.

> pmdarima는 **stepwise + Python 순차 실행**이라 `s=24` 시계열에서 매우 느림. 전체 3년(n=26,279)은 수 시간 단위라 여기서는 **최근 60일(1,440h)** subset으로 비교.

stepwise=True (기본값) — Hyndman-Khandakar stepwise 탐색
- AR/MA 차수를 한 번에 하나씩 조정하면서 "더 좋아지는 방향"으로만 이동하는 탐욕적(greedy) 탐색입니다. 
- 기준 모델에서 시작해서 p, q, P, Q를 ±1씩 바꿔보고, 정보기준(AIC/BIC/HQIC)이 개선되면 그 모델을 새 기준으로 삼아 다시 주변을 탐색하는 방식입니다. pmdarima와 동일한 로직이에요.  
   
stepwise=False — Exhaustive grid search (Rayon 병렬)  
- max_p, max_q, max_P, max_Q 범위 안의 모든 차수 조합을 전부 적합해보고 그중 최적을 고릅니다. 
- rustima는 이걸 Rayon work-stealing 스레드 풀로 병렬 처리해서 조합 하나당 전체 fit job을 워커에 분산시켜요.


In [10]:
# ── [A] rustima auto_arima 단독 실행 ─────────────────────────
from rustima import auto_arima as rs_auto

t0 = time.perf_counter()
auto_rs = rs_auto(y_paju, s=24, trend='n', stepwise=True, trace=False)
t_rs = time.perf_counter() - t0

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

print(f"rustima  : {rs_order}{rs_seasonal}  AIC={rs_aic:.2f}  time={t_rs:.3f}s")

rustima  : (3, 0, 3)(2, 1, 2, 24)  AIC=162592.42  time=417.780s


In [17]:
# ── [B] pmdarima auto_arima 단독 실행 ────────────────────────
import pmdarima as pm

t0 = time.perf_counter()
auto_pm = pm.auto_arima(
    y_paju,
    seasonal=True, m=24, trend=None,
    stepwise=True, suppress_warnings=True,
)
t_pm = time.perf_counter() - t0

pm_order    = auto_pm.order
pm_seasonal = auto_pm.seasonal_order
pm_aic      = float(auto_pm.aic())

print(f"pmdarima : {pm_order}{pm_seasonal}  AIC={pm_aic:.2f}  time={t_pm:.3f}s")

: 

In [ ]:
# ── [C] 최종 결과 비교 ───────────────────────────────────────
speedup  = t_pm / t_rs if t_rs > 0 else float('inf')
aic_diff = pm_aic - rs_aic   # 음수면 pmdarima가 더 낮은 AIC

print("="*64)
print(f"{'engine':<10}{'order':<14}{'seasonal':<16}{'AIC':>12}{'time(s)':>12}")
print("-"*64)
print(f"{'rustima':<10}{str(rs_order):<14}{str(rs_seasonal):<16}{rs_aic:>12.2f}{t_rs:>12.3f}")
print(f"{'pmdarima':<10}{str(pm_order):<14}{str(pm_seasonal):<16}{pm_aic:>12.2f}{t_pm:>12.3f}")
print("="*64)
print(f"속도   : rustima 가 {speedup:.1f}x 빠름  (절감 {t_pm - t_rs:.2f}s)")
print(f"AIC 차 : pmdarima - rustima = {aic_diff:+.2f}  "
      f"({'동일' if abs(aic_diff) < 1e-2 else ('rustima 우세' if aic_diff > 0 else 'pmdarima 우세')})")
print(f"order  : {'동일' if rs_order == pm_order and rs_seasonal == pm_seasonal else '다름'}")

## rusima 다른 방법으로도 탐색

In [8]:
from rustima import auto_arima as rs_auto

### 1. grid_search, trend = "n"

In [9]:
t0 = time.perf_counter()
auto_rs = rs_auto(y_paju, s=24, trend='n', stepwise=False, trace=False)
t_rs = time.perf_counter() - t0

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

print(f"rustima  : {rs_order}{rs_seasonal}  AIC={rs_aic:.2f}  time={t_rs:.3f}s")

rustima  : (3, 0, 5)(2, 1, 2, 24)  AIC=162558.83  time=1209.534s


In [10]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {t_rs:.2f} s  ({len(auto_rs.history)} 모델 평가)")
print('=' * 60)
print(auto_rs.search_summary())
print()
print("[Best model summary]")
print(auto_rs.result.summary())


auto_arima 소요 시간: 1209.53 s  (324 모델 평가)
auto_arima: Best ARIMA(3,0,5)(2,1,2)[24]
  aic=162558.827
  Models evaluated: 324 (322 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(3,0,5)(2,1,2)[24]                  Log Likelihood:   -81266.413
No. Observations: 26279                            AIC:             162558.827
Trend: n                                           BIC:             162665.121
Method: lbfgsb                                     HQIC:            162593.149
Converged: True                                     Scale:           28.533927
Date: 2026-04-22                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
           ar.L1     0.7883
           ar.L2     0.9014
           ar.L3    -0.6923
           ma.L

### 2. grid_search, trend = "c"

In [11]:
t0 = time.perf_counter()
auto_rs = rs_auto(y_paju, s=24, trend='c', stepwise=False, trace=False)
t_rs = time.perf_counter() - t0

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

print(f"rustima  : {rs_order}{rs_seasonal}  AIC={rs_aic:.2f}  time={t_rs:.3f}s")

rustima  : (3, 0, 5)(2, 1, 2, 24)  AIC=162564.37  time=1467.842s


In [12]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {t_rs:.2f} s  ({len(auto_rs.history)} 모델 평가)")
print('=' * 60)
print(auto_rs.search_summary())
print()
print("[Best model summary]")
print(auto_rs.result.summary())


auto_arima 소요 시간: 1467.84 s  (324 모델 평가)
auto_arima: Best ARIMA(3,0,5)(2,1,2)[24]
  aic=162564.367
  Models evaluated: 324 (324 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(3,0,5)(2,1,2)[24]                  Log Likelihood:   -81268.184
No. Observations: 26279                            AIC:             162564.367
Trend: c                                           BIC:             162678.839
Method: lbfgsb                                     HQIC:            162601.330
Converged: True                                     Scale:           28.537770
Date: 2026-04-22                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
       intercept    -0.0000
           ar.L1     0.7965
           ar.L2     0.8748
           ar.L

### 3. grid_search, trend = "t"

In [13]:
t0 = time.perf_counter()
auto_rs = rs_auto(y_paju, s=24, trend='t', stepwise=False, trace=False)
t_rs = time.perf_counter() - t0

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

print(f"rustima  : {rs_order}{rs_seasonal}  AIC={rs_aic:.2f}  time={t_rs:.3f}s")

rustima  : (5, 0, 0)(2, 1, 2, 24)  AIC=163447.16  time=482.217s


In [14]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {t_rs:.2f} s  ({len(auto_rs.history)} 모델 평가)")
print('=' * 60)
print(auto_rs.search_summary())
print()
print("[Best model summary]")
print(auto_rs.result.summary())


auto_arima 소요 시간: 482.22 s  (324 모델 평가)
auto_arima: Best ARIMA(5,0,0)(2,1,2)[24]
  aic=163447.163
  Models evaluated: 324 (324 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(5,0,0)(2,1,2)[24]                  Log Likelihood:   -81712.582
No. Observations: 26279                            AIC:             163447.163
Trend: t                                           BIC:             163537.105
Method: lbfgsb                                     HQIC:            163476.205
Converged: True                                     Scale:           29.530664
Date: 2026-04-22                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
           drift     0.0000
           ar.L1     1.0817
           ar.L2    -0.2411
           ar.L3

### 4. grid_search, trend = "ct"

In [15]:
t0 = time.perf_counter()
auto_rs = rs_auto(y_paju, s=24, trend='ct', stepwise=False, trace=False)
t_rs = time.perf_counter() - t0

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

print(f"rustima  : {rs_order}{rs_seasonal}  AIC={rs_aic:.2f}  time={t_rs:.3f}s")

rustima  : (4, 0, 5)(2, 1, 2, 24)  AIC=162863.91  time=431.290s


In [16]:
print()
print('=' * 60)
print(f"auto_arima 소요 시간: {t_rs:.2f} s  ({len(auto_rs.history)} 모델 평가)")
print('=' * 60)
print(auto_rs.search_summary())
print()
print("[Best model summary]")
print(auto_rs.result.summary())


auto_arima 소요 시간: 431.29 s  (324 모델 평가)
auto_arima: Best ARIMA(4,0,5)(2,1,2)[24]
  aic=162863.911
  Models evaluated: 324 (324 converged)

[Best model summary]
                               SARIMAX Results                                
Model: SARIMAX(4,0,5)(2,1,2)[24]                  Log Likelihood:   -81415.955
No. Observations: 26279                            AIC:             162863.911
Trend: ct                                          BIC:             162994.735
Method: lbfgsb                                     HQIC:            162906.153
Converged: True                                     Scale:           28.868288
Date: 2026-04-22                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
       intercept     0.0056
           drift    -0.0000
           ar.L1     1.4663
           ar.L2